# One CNN-autoencoder run, end to end

This notebook runs a single **CNN autoencoder** experiment step by step, with a plot at every step:

1. settings and config
2. data: load, split, inspect (observed flux, target tellurics, metadata, noise)
3. model: build, inspect, untrained prediction
4. training
5. loss curves
6. evaluation on the test nights
7. real observations (if `data.real` is set)

**Model:** one observation in, one telluric spectrum out —
`observed (B, 2048) + metadata [airmass, pwv] (B, 2) → telluric (B, 2048)`, trained with MSE on the telluric transmission.

**Before running on your data:** set `$TELLURICS_DATA` and the paths in
`experiments/single_frame/experiment_cnn_ae.yaml` (`night_h5`, `phoenix_dir`, `observed_key`, `real`).
To try the notebook without any data, set `USE_SYNTHETIC = True` below: it builds a small synthetic dataset in a temporary folder.

## 1. Settings

In [ ]:
from pathlib import Path
import os, sys, tempfile

import numpy as np
import torch
import matplotlib.pyplot as plt

# The notebook lives in tests/: find the repo root and make the src-layout package importable.
_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd if (_cwd / "experiments").exists() else _cwd.parent
for p in (REPO_ROOT / "src", REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# ---- settings ------------------------------------------------------------- #
CONFIG = REPO_ROOT / "experiments" / "single_frame" / "experiment_cnn_ae.yaml"
USE_SYNTHETIC = False      # True: build a small synthetic dataset (no real data needed)
MAX_EPOCHS = 5             # short notebook run; the YAML's max_epochs is for full runs
DEVICE = "auto"            # "auto" | "cpu" | "gpu"
OVERRIDES = []             # extra dotted overrides, e.g. ["model.latent_dim=64", "data.noise_sigma=0.05"]
RUN_DIR = REPO_ROOT / "runs" / "notebook_cnn_ae"

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
torch.manual_seed(0); np.random.seed(0)
print("repo root:", REPO_ROOT)
print("config   :", CONFIG.relative_to(REPO_ROOT))

### 1b. (Optional) synthetic data

Only runs with `USE_SYNTHETIC = True`. It writes, in a temporary folder:

* `sim.h5` — flat layout (one row per observation, nights with different numbers of frames):
  `transmission` (the target), `observed` = transmission × a smooth star, `wavelength`, labels `series_index`, `airmass`, `pwv`;
* `real.h5` — "real" observations: `observed` flux only, **no** target.

and points the config at them (`observed_key: observed`, so no PHOENIX files are needed). The demo trains for 30 epochs at
learning rate 1e-3 (about a minute on a CPU), long enough for the model to learn the absorption lines.

In [ ]:
if USE_SYNTHETIC:
    import h5py

    def write_synthetic(path, n_nights, rng, with_target=True, n_wave=2048):
        wl = np.linspace(2000.0, 2100.0, n_wave)                      # nm
        centres = rng.uniform(2000, 2100, 60)                          # telluric line positions (fixed)
        strengths = rng.uniform(0.2, 1.0, 60)
        profile = sum(s * np.exp(-0.5 * ((wl - c) / 0.04) ** 2) for c, s in zip(centres, strengths))
        rows, series = [], []
        for night in range(n_nights):
            pwv = rng.uniform(1, 20)                                   # mm, fixed per night
            star = 1 + 0.1 * np.sin(wl / rng.uniform(5, 15)) - 0.3 * np.exp(-0.5 * ((wl - rng.uniform(2000, 2100)) / 0.3) ** 2)
            for k in range(rng.integers(5, 15)):                       # unequal frames per night
                airmass = 1.0 + 0.9 * k / 15
                trans = np.exp(-0.08 * airmass * pwv * profile)
                rows.append((trans, trans * star, airmass, pwv)); series.append(night)
        trans, obs, airmass, pwv = (np.array(x) for x in zip(*rows))
        labels = np.column_stack([series, airmass, pwv]).astype(np.float32)
        with h5py.File(path, "w") as f:
            if with_target:
                f.create_dataset("transmission", data=trans.astype(np.float32))
            f.create_dataset("observed", data=obs.astype(np.float32))
            f.create_dataset("wavelength", data=wl)
            f.create_dataset("labels", data=labels).attrs["columns"] = np.array(["series_index", "airmass", "pwv"], dtype="S")

    syn_root = Path(tempfile.mkdtemp(prefix="tellurics_synthetic_"))
    rng = np.random.default_rng(1)
    write_synthetic(syn_root / "sim.h5", n_nights=60, rng=rng)
    write_synthetic(syn_root / "real.h5", n_nights=3, rng=np.random.default_rng(1), with_target=False)
    os.environ["TELLURICS_DATA"] = str(syn_root)
    # The small synthetic set learns fast enough to show the lines in ~1 min on CPU.
    MAX_EPOCHS = 30
    OVERRIDES = [
        "training.optimizer.learning_rate=1e-3",
        "data.night_h5=sim.h5", "data.observed_key=observed", "data.max_nights=null",
        "data.real=[{name: synthetic_real, night_h5: real.h5, observed_key: observed}]",
        *OVERRIDES,
    ]
    print("synthetic data in", syn_root, "| synthetic demo: 30 epochs, lr 1e-3")

## 2. Config

In [ ]:
from tellurics.config import load_config
from tellurics.scripts.train import apply_overrides

config = apply_overrides(load_config(CONFIG), [*OVERRIDES, f"training.max_epochs={MAX_EPOCHS}"])
d, m, t = config.data, config.model, config.training
print(f"architecture : {m.architecture.value}  (N = {m.num_wavelength_bins})")
print(f"data file    : {d.night_h5}")
print(f"input        : {'observed flux from ' + repr(d.observed_key) if d.observed_key else 'tellurics x star (phoenix_dir)'}")
print(f"noise        : {d.noise_sigma:.0%} per pixel" + (" (simulations)" if d.simulated else " (off: not simulated)"))
print(f"metadata     : {d.metadata_columns}  bounds {d.bounds}")
print(f"real files   : {[r.name for r in d.real] or 'none'}")
print(f"training     : {t.max_epochs} epochs, batch {t.batch_size}, lr {t.optimizer.learning_rate}")

## 3. Data

The data module splits **by night** (no frame of a validation/test night is seen in training), assigns one star per night
(for pure-telluric files) and adds the Gaussian noise to simulated inputs (fresh in training, fixed in validation/test).

In [ ]:
from tellurics.data.datamodule import TelluricDataModule

RUN_DIR.mkdir(parents=True, exist_ok=True)
dm = TelluricDataModule(d, run_dir=RUN_DIR, batch_size=t.batch_size,
                        num_workers=0, single_frame=m.single_frame)
dm.setup()
wl = dm.wavegrid.wavelength
for name, ds, ids in (("train", dm.train_ds, dm.train_ids), ("val", dm.val_ds, dm.val_ids), ("test", dm.test_ds, dm.test_ids)):
    print(f"{name:5s}: {len(ids):4d} nights -> {len(ds):6d} observations")
for name, ds in dm.real_ds.items():
    print(f"real '{name}': {len(ds)} observations (no target)")

### 3a. One observation: input vs target

In [ ]:
sample = dm.val_ds[0]                        # validation: fixed noise
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(wl, sample["observed"], lw=0.7, label="observed (model input, with noise)")
if "stellar" in sample:
    axes[0].plot(wl, sample["stellar"], lw=0.8, alpha=0.8, label="stellar spectrum S")
axes[0].set_ylabel("flux"); axes[0].legend(loc="lower left")
axes[1].plot(wl, sample["telluric"], lw=0.7, color="C2", label="telluric transmission (target)")
axes[1].set_ylabel("transmission"); axes[1].set_xlabel(f"wavelength [{dm.wavegrid.unit}]"); axes[1].legend(loc="lower left")
axes[0].set_title(f"night {sample['night_id']}, frame {sample['frame_id']}  |  metadata (scaled) = {sample['theta']['metadata'].numpy().round(3)}")
plt.tight_layout(); plt.show()

### 3b. Metadata per split and frames per night

In [ ]:
def metadata_of(ds, n=2000):
    idx = np.linspace(0, len(ds) - 1, min(n, len(ds))).astype(int)
    return np.stack([ds[i]["theta"]["metadata"].numpy() for i in idx])

cols = d.metadata_columns
fig, axes = plt.subplots(1, len(cols) + 1, figsize=(4.2 * (len(cols) + 1), 3.3))
for name, ds in (("train", dm.train_ds), ("val", dm.val_ds), ("test", dm.test_ds)):
    meta = metadata_of(ds)
    for j, c in enumerate(cols):
        axes[j].hist(meta[:, j], bins=30, alpha=0.5, density=True, label=name)
for j, c in enumerate(cols):
    axes[j].set_title(f"{c} (scaled to [0, 1])"); axes[j].legend()
if dm.night_rows is not None:
    counts = [len(r) for r in dm.night_rows.values()]
    axes[-1].hist(counts, bins=np.arange(min(counts), max(counts) + 2) - 0.5)
    axes[-1].set_title("frames per night"); axes[-1].set_xlabel("frames")
else:
    axes[-1].text(0.5, 0.5, f"{dm.n_frames} frames every night", ha="center"); axes[-1].set_axis_off()
plt.tight_layout(); plt.show()

### 3c. The input noise (simulations only)

In [ ]:
if d.simulated and d.noise_sigma > 0:
    ds = dm.train_ds
    noisy = np.stack([ds[i]["observed"].numpy() for i in range(min(64, len(ds)))])
    sigma, ds.noise_sigma = ds.noise_sigma, 0.0            # same samples without noise
    clean = np.stack([ds[i]["observed"].numpy() for i in range(min(64, len(ds)))])
    ds.noise_sigma = sigma
    rel = ((noisy - clean) / clean).ravel()
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.3))
    axes[0].plot(wl, clean[0], lw=0.7, label="clean"); axes[0].plot(wl, noisy[0], lw=0.5, alpha=0.7, label="noisy")
    axes[0].set_xlim(wl[0], wl[0] + (wl[-1] - wl[0]) / 10); axes[0].legend(); axes[0].set_title("zoom: clean vs noisy input")
    axes[1].hist(rel, bins=80, density=True); axes[1].set_title(f"(noisy - clean) / clean: std = {rel.std():.3f} (target {d.noise_sigma})")
    plt.tight_layout(); plt.show()
else:
    print("no input noise (data.noise_sigma = 0 or data.simulated = false)")

## 4. Model

In [ ]:
from tellurics.scripts.train import build_model

module = build_model(m, t)            # LightningModule wrapping the CNN autoencoder
counts = module.model.parameter_counts()
for name, n in counts.items():
    print(f"  {name:20s} {n:>10,}")
print(f"  {'total':20s} {module.model.count_parameters():>10,}")

batch = next(iter(dm.val_dataloader()))
print({k: tuple(v.shape) for k, v in batch.items() if torch.is_tensor(v)}, "metadata", tuple(batch["theta"]["metadata"].shape))
with torch.no_grad():
    out = module._run_estimator(batch)
print("prediction:", tuple(out.telluric.shape), " latent:", tuple(out.latent.shape))

plt.figure(figsize=(11, 3))
plt.plot(wl, batch["telluric"][0], lw=0.7, label="target")
plt.plot(wl, out.telluric[0], lw=0.9, label="untrained prediction (clear-sky start)")
plt.legend(); plt.title("before training"); plt.xlabel(f"wavelength [{dm.wavegrid.unit}]"); plt.show()

## 5. Training

A short run (`MAX_EPOCHS`). Losses are recorded per epoch; the best checkpoint (lowest `validation_loss`) is kept in `RUN_DIR`.

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint

class History(pl.Callback):
    """Collect the epoch-level metrics for plotting."""
    def __init__(self):
        self.train, self.val = [], []
    def on_validation_epoch_end(self, trainer, pl_module):
        if not trainer.sanity_checking:
            self.val.append(float(trainer.callback_metrics["validation_loss"]))
    def on_train_epoch_end(self, trainer, pl_module):
        self.train.append(float(trainer.callback_metrics["training_loss"]))

history = History()
checkpoint = ModelCheckpoint(dirpath=RUN_DIR / "checkpoints", monitor="validation_loss", mode="min",
                             filename="best-{epoch:03d}-{validation_loss:.5f}")
trainer = pl.Trainer(
    max_epochs=t.max_epochs, accelerator="cpu" if DEVICE == "cpu" else "auto",
    callbacks=[history, checkpoint], logger=False, enable_model_summary=False,
    gradient_clip_val=t.gradient_clip_val, log_every_n_steps=10,
)
trainer.fit(module, datamodule=dm)
print("best checkpoint:", checkpoint.best_model_path, " val loss:", float(checkpoint.best_model_score))

## 6. Loss curves

In [ ]:
epochs = np.arange(1, len(history.train) + 1)
plt.figure(figsize=(7, 3.5))
plt.semilogy(epochs, history.train, "o-", label="training")
plt.semilogy(epochs, history.val[: len(epochs)], "s-", label="validation")
plt.xlabel("epoch"); plt.ylabel("MSE (telluric transmission)"); plt.legend(); plt.title("loss")
plt.show()

## 7. Evaluation on the test nights (best checkpoint)

In [ ]:
from tellurics.training.predict import load_trained_module, predict_loader

best = load_trained_module(checkpoint.best_model_path, map_location="cpu")
test_loader = dm.test_dataloader()
pred = predict_loader(best, test_loader)["telluric_pred"]                 # (n, N)
truth = np.concatenate([b["telluric"].numpy() for b in test_loader])
meta = np.concatenate([b["theta"]["metadata"].numpy() for b in test_loader])
res = pred - truth
print(f"test MSE {np.mean(res**2):.3e}   RMSE {np.sqrt(np.mean(res**2)):.3e}   max |res| {np.abs(res).max():.3e}")

# a few test observations: prediction vs truth, and residuals
rows = np.linspace(0, len(pred) - 1, 3).astype(int)
fig, axes = plt.subplots(len(rows), 1, figsize=(11, 2.4 * len(rows)), sharex=True)
for ax, i in zip(axes, rows):
    ax.plot(wl, truth[i], lw=0.8, label="true telluric")
    ax.plot(wl, pred[i], lw=0.8, ls="--", label="predicted")
    ax2 = ax.twinx(); ax2.plot(wl, res[i], lw=0.5, color="C3", alpha=0.6); ax2.set_ylabel("residual", color="C3")
    ax.set_ylabel("T"); ax.set_title(f"test observation {i}", fontsize=9)
axes[0].legend(loc="lower left"); axes[-1].set_xlabel(f"wavelength [{dm.wavegrid.unit}]")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
axes[0].plot(wl, np.sqrt(np.mean(res**2, axis=0)), lw=0.7)
axes[0].set_title("RMSE per wavelength"); axes[0].set_xlabel(f"wavelength [{dm.wavegrid.unit}]")
axes[1].hist(res.ravel(), bins=100, log=True); axes[1].set_title("residuals (all pixels)")
rmse_obs = np.sqrt(np.mean(res**2, axis=1))
for j, c in enumerate(d.metadata_columns[:2]):
    axes[2].scatter(meta[:, j], rmse_obs, s=6, alpha=0.6, label=c)
axes[2].set_xlabel("metadata (scaled)"); axes[2].set_ylabel("RMSE per observation"); axes[2].legend()
axes[2].set_title("error vs airmass / pwv")
plt.tight_layout(); plt.show()

## 8. Real observations

Only if the config lists `data.real`. These have no telluric target: the model predicts their tellurics, and dividing the
observed flux by the prediction gives the telluric-corrected spectrum. Predictions are also saved to `RUN_DIR/predictions/<name>.h5`.

In [ ]:
from tellurics.training.predict import predict_real_observations

if dm.real_ds:
    written = predict_real_observations(best, dm, RUN_DIR)
    for name, loader in dm.real_dataloaders().items():
        out = predict_loader(best, loader)
        i = 0
        fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
        axes[0].plot(wl, out["observed"][i], lw=0.7, label="observed")
        axes[0].plot(wl, out["telluric_pred"][i] * np.median(out["observed"][i]), lw=0.7, alpha=0.8, label="predicted telluric (scaled)")
        axes[0].legend(loc="lower left"); axes[0].set_title(f"real '{name}', observation {i}")
        axes[1].plot(wl, out["observed"][i] / out["telluric_pred"][i], lw=0.7, color="C2", label="observed / predicted telluric")
        axes[1].legend(loc="lower left"); axes[1].set_xlabel(f"wavelength [{dm.wavegrid.unit}]")
        plt.tight_layout(); plt.show()
    print("saved:", *written, sep="\n  ")
else:
    print("no data.real observations in the config")

## Summary

* data: night-level split, one star per night (pure-telluric files) or stored observed flux, noise on simulations only;
* model: CNN autoencoder, `(B, 2048) + metadata (B, 2) → (B, 2048)`;
* the best checkpoint is in `RUN_DIR/checkpoints/`, real-observation predictions in `RUN_DIR/predictions/`.

For full runs use the training script (`python -m tellurics.scripts.train --config experiments/single_frame/experiment_cnn_ae.yaml`)
or the sweep (`python -m src.tellurics.scripts.sweep_cnn_ae`).